In [ ]:
namespace = "default"
ray_image = "has to be specified"
openshift_api_url = "has to be specified"
kubernetes_user_bearer_token = "has to be specified"
num_gpus = 1
script_configmap = "default"
algorithm = "sft"
pvc_name = "default"

In [ ]:
import os
import subprocess
import time

from codeflare_sdk import RayJob, ClusterConfiguration
from kubernetes.client import V1Volume, V1VolumeMount, V1ConfigMapVolumeSource

num_gpus = int(num_gpus)

In [ ]:
result = subprocess.run(
    ["oc", "login", f"--token={kubernetes_user_bearer_token}",
     f"--server={openshift_api_url}", "--insecure-skip-tls-verify"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError(f"oc login failed: {result.stderr}")

In [ ]:
import json
import socket

PVC_MOUNT_PATH = "/opt/app-root/src"
MODEL_ID = "allenai/OLMo-2-0425-1B-Instruct"
MODEL_S3_KEY = "models/OLMo-2-0425-1B-Instruct"
MODEL_DIR = f"{PVC_MOUNT_PATH}/model"
DATASET_DIR = f"{PVC_MOUNT_PATH}/dataset"

socket.setdefaulttimeout(10)

try:
    import s3fs
    HAS_S3FS = True
except ImportError:
    HAS_S3FS = False

s3_endpoint = os.environ.get("AWS_DEFAULT_ENDPOINT", "")
s3_access_key = os.environ.get("AWS_ACCESS_KEY_ID", "")
s3_secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY", "")
s3_bucket = os.environ.get("AWS_STORAGE_BUCKET", "")
s3_prefix = os.environ.get("AWS_STORAGE_BUCKET_RAY_TRAINING_HUB_DIR", "")

model_downloaded = False

if HAS_S3FS and s3_endpoint and s3_bucket:
    try:
        endpoint_url = s3_endpoint if s3_endpoint.startswith("http") else f"https://{s3_endpoint}"
        prefix = (s3_prefix or "").strip("/")
        print(f"[notebook] S3 configured: endpoint={endpoint_url}, bucket={s3_bucket}, prefix={prefix or '<root>'}")

        fs = s3fs.S3FileSystem(
            key=s3_access_key,
            secret=s3_secret_key,
            endpoint_url=endpoint_url,
            use_ssl=endpoint_url.startswith("https"),
            config_kwargs={"signature_version": "s3v4"},
            client_kwargs={"verify": False},
        )

        model_remote = f"{s3_bucket}/{prefix}/{MODEL_S3_KEY}" if prefix else f"{s3_bucket}/{MODEL_S3_KEY}"
        os.makedirs(MODEL_DIR, exist_ok=True)
        print(f"[notebook] Downloading model from s3://{model_remote} to {MODEL_DIR}...")
        for remote_file in fs.find(model_remote):
            rel = remote_file[len(model_remote):].lstrip("/")
            if not rel or remote_file.endswith("/"):
                continue
            dst = os.path.join(MODEL_DIR, rel)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            if not os.path.exists(dst):
                fs.get(remote_file, dst)
        if os.listdir(MODEL_DIR):
            model_downloaded = True
            print(f"[notebook] Model downloaded from S3 to {MODEL_DIR}")
        else:
            print("[notebook] S3 model directory was empty")

        if algorithm == "grpo":
            dataset_remote = f"{s3_bucket}/{prefix}/grpo-dataset" if prefix else f"{s3_bucket}/grpo-dataset"
            os.makedirs(DATASET_DIR, exist_ok=True)
            print(f"[notebook] Downloading GRPO dataset from s3://{dataset_remote}...")
            for remote_file in fs.find(dataset_remote):
                rel = remote_file[len(dataset_remote):].lstrip("/")
                if not rel or remote_file.endswith("/"):
                    continue
                dst = os.path.join(DATASET_DIR, rel)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                if not os.path.exists(dst):
                    fs.get(remote_file, dst)
            print(f"[notebook] GRPO dataset downloaded from S3 to {DATASET_DIR}")

    except Exception as e:
        print(f"[notebook] S3 download failed: {e}")
        import traceback
        traceback.print_exc()
        print("[notebook] Will attempt HuggingFace fallback...")
else:
    if not HAS_S3FS:
        print("[notebook] S3 not available: s3fs not installed")
    else:
        print("[notebook] S3 not configured (missing endpoint or bucket env vars)")

if not model_downloaded:
    print(f"[notebook] Downloading model {MODEL_ID} from HuggingFace...")
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=MODEL_ID, local_dir=MODEL_DIR)
    print(f"[notebook] Model downloaded to {MODEL_DIR}")

    if algorithm == "grpo":
        from datasets import load_dataset, DownloadConfig

        DATASET_ID = "Agent-Ark/Toucan-1.5M"
        GRPO_SUBSET_SIZE = 200
        print(f"[notebook] Loading {GRPO_SUBSET_SIZE}-sample subset of {DATASET_ID}...")
        dataset = load_dataset(DATASET_ID, "Qwen3", data_files="Qwen3/train-00000-of-00044.parquet", split=f"train[:{GRPO_SUBSET_SIZE}]", verification_mode="no_checks")
        os.makedirs(DATASET_DIR, exist_ok=True)
        dataset.to_json(f"{DATASET_DIR}/train.jsonl")
        print(f"[notebook] Dataset subset ({len(dataset)} samples) saved to {DATASET_DIR}/train.jsonl")

print(f"[notebook] Model ready at {MODEL_DIR}")

if algorithm == "grpo":
    # vLLM selects mistral-common's sentencepiece backend when
    # tokenizer.model.v3 is present, but the image has no
    # sentencepiece. tokenizer.json provides the same tokenizer
    # through the standard HF fast-tokenizer path.
    # NB: only v3 -- vLLM selects the mistral-common backend via the
    # "tokenizer.model.v*" file pattern; plain "tokenizer.model" does
    # not match it and must be kept.
    for _tf in ("tokenizer.model.v3",):
        _tp = os.path.join(MODEL_DIR, _tf)
        if os.path.exists(_tp):
            os.remove(_tp)
            print(f"[notebook] Removed {_tf} (mistral-common backend unavailable; using tokenizer.json)")
    # snapshot_download leaves .cache/huggingface/download/<file>.metadata
    # inside local_dir; "tokenizer.model.v3.metadata" matches vLLM's
    # "tokenizer.model.v*" backend-selection pattern, so it must go too.
    import shutil as _shutil
    _cache_dir = os.path.join(MODEL_DIR, ".cache")
    if os.path.isdir(_cache_dir):
        _shutil.rmtree(_cache_dir)
        print("[notebook] Removed snapshot metadata cache from model dir")
    # verl's agent loop probes the chat template with two consecutive user
    # messages (initialize_system_prompt / initialize_turn_separator
    # token-diff arithmetic); Mistral's template raises on non-alternating
    # roles, killing every generation worker. The converted dataset
    # alternates correctly, so the guard is redundant here -- neutralize
    # only that check (the tool-call-ID check is kept; our IDs comply).
    _tc_path = os.path.join(MODEL_DIR, "tokenizer_config.json")
    if os.path.exists(_tc_path):
        with open(_tc_path) as _f:
            _tc = json.load(_f)
        _tmpl = _tc.get("chat_template", "")
        _bad = ('raise_exception("After the optional system message, conversation '
                'roles must alternate user/assistant/user/assistant/...")')
        if _bad in _tmpl:
            _tc["chat_template"] = _tmpl.replace(_bad, '""')
            with open(_tc_path, "w") as _f:
                json.dump(_tc, _f)
            print("[notebook] Relaxed chat-template role-alternation check (verl probe compatibility)")

if algorithm == "grpo":
    # Mistral's chat template requires tool-call IDs to be exactly 9
    # alphanumeric characters and only understands the modern
    # tool_calls/tool message format. The dataset stores messages as a
    # JSON-encoded string in the deprecated function_call/function format
    # (no IDs); training_hub converts that at runtime but mints
    # "call_" + 24-hex IDs, which Mistral's template rejects. Convert to
    # tool_calls/tool here with 9-char IDs so training_hub passes them
    # through untouched.
    import hashlib
    import json

    def _mistral_tool_call_id(original_id):
        # Deterministic 9-char alphanumeric (sha256 prefix keeps paired
        # assistant/tool messages referencing the same ID).
        return "x" + hashlib.sha256(str(original_id).encode()).hexdigest()[:8]

    def _convert_message(msg, last_tool_call_id):
        if not isinstance(msg, dict):
            return msg, last_tool_call_id
        if msg.get("function_call"):
            fc = msg.pop("function_call")
            last_tool_call_id = _mistral_tool_call_id(
                f"{fc.get('name')}:{fc.get('arguments')}"
            )
            msg["tool_calls"] = [{
                "type": "function",
                "id": last_tool_call_id,
                "function": {
                    "name": fc.get("name"),
                    "arguments": fc.get("arguments", "{}"),
                },
            }]
        elif msg.get("role") == "function":
            msg["role"] = "tool"
            msg["tool_call_id"] = last_tool_call_id or _mistral_tool_call_id("orphan")
        # Normalize any pre-existing tool-call IDs to 9 chars.
        tool_calls = msg.get("tool_calls")
        if isinstance(tool_calls, str):
            try:
                tool_calls = json.loads(tool_calls)
            except ValueError:
                tool_calls = []
        if isinstance(tool_calls, list):
            normalized = []
            for tc in tool_calls:
                if isinstance(tc, str):
                    try:
                        tc = json.loads(tc)
                    except ValueError:
                        continue
                if isinstance(tc, dict):
                    if tc.get("id"):
                        tc["id"] = _mistral_tool_call_id(tc["id"])
                    normalized.append(tc)
            if normalized:
                msg["tool_calls"] = normalized
        if msg.get("tool_call_id"):
            msg["tool_call_id"] = _mistral_tool_call_id(msg["tool_call_id"])
        return msg, last_tool_call_id

    dataset_file = f"{DATASET_DIR}/train.jsonl"
    if os.path.exists(dataset_file):
        records, converted = [], 0
        with open(dataset_file) as f:
            for line in f:
                if not line.strip():
                    continue
                rec = json.loads(line)
                msgs = rec.get("messages")
                if isinstance(msgs, str):
                    try:
                        msgs = json.loads(msgs)
                    except ValueError:
                        msgs = None
                if isinstance(msgs, list):
                    out, last_id = [], None
                    for msg in msgs:
                        msg, last_id = _convert_message(msg, last_id)
                        out.append(msg)
                    # Keep the source's JSON-string encoding of messages.
                    rec["messages"] = json.dumps(out)
                    converted += 1
                records.append(rec)
        with open(dataset_file, "w") as f:
            for rec in records:
                f.write(json.dumps(rec) + "\n")
        print(f"[notebook] Converted {converted} traces to tool_calls format with 9-char IDs (Mistral template requirement)")
if algorithm == "grpo":
    print(f"[notebook] Dataset ready at {DATASET_DIR}")

In [ ]:
from kubernetes.client import V1PersistentVolumeClaimVolumeSource

SCRIPT_MOUNT_PATH = "/mnt/scripts"
SHARED_DATA_MOUNT_PATH = "/mnt/shared"

env_vars = {
    "MODEL_DIR": f"{SHARED_DATA_MOUNT_PATH}/model",
    "GRPO_N_GPUS": str(num_gpus),
}
if algorithm == "grpo":
    env_vars["GRPO_DATA_PATH"] = f"{SHARED_DATA_MOUNT_PATH}/dataset/train.jsonl"
    # Full fine-tuning, not LoRA: released verl (0.9.x) has a broken
    # LoRA weight-sync path (layered_summon returns empty and the
    # fallback sends PEFT-named tensors that vLLM add_lora maps to
    # zero layers -> StopIteration). lora_r=0 takes the non-PEFT sync
    # path, which streams per-tensor and avoids the bug entirely.
    env_vars["GRPO_LORA_R"] = "0"
hf_token = os.environ.get("HF_TOKEN", "")
if hf_token:
    env_vars["HF_TOKEN"] = hf_token
    env_vars["HUGGING_FACE_HUB_TOKEN"] = hf_token

if algorithm == "grpo":
    head_accels = {}
    n_workers = 1
    worker_accels = {"nvidia.com/gpu": num_gpus}
    # verl pins 4 transfer_queue storage units to the worker node, and its
    # placement groups need up to 14 more CPUs there (2x {CPU:3, GPU:1}
    # training bundles + an 8-CPU resource-pool group on verl 0.9.0).
    worker_cpu_req, worker_cpu_lim = 4, 20
    # 7B model: the initial weight sync peaks at ~175Gi on the worker
    # node (2x 51Gi WorkerDict full-state-dict + 2x 29Gi vLLM workers,
    # per Ray OOM report), so the limit must clear that with headroom.
    worker_mem_req, worker_mem_lim = 96, 256
    head_cpu_req, head_cpu_lim = 2, 16
else:
    head_accels = {"nvidia.com/gpu": num_gpus}
    n_workers = 0
    worker_accels = {}
    worker_cpu_req, worker_cpu_lim = 1, 1
    worker_mem_req, worker_mem_lim = 2, 2
    head_cpu_req, head_cpu_lim = 2, 4

cluster_config = ClusterConfiguration(
    name=f"{algorithm}-e2e-test-cluster",
    image=ray_image,
    head_cpu_requests=head_cpu_req,
    head_cpu_limits=head_cpu_lim,
    head_memory_requests=32,
    head_memory_limits=48,
    head_extended_resource_requests=head_accels,
    num_workers=n_workers,
    worker_cpu_requests=worker_cpu_req,
    worker_cpu_limits=worker_cpu_lim,
    worker_memory_requests=worker_mem_req,
    worker_memory_limits=worker_mem_lim,
    worker_extended_resource_requests=worker_accels,
    envs=env_vars,
    volumes=[
        V1Volume(
            name="train-scripts",
            config_map=V1ConfigMapVolumeSource(name=script_configmap)
        ),
        V1Volume(
            name="shared-data",
            persistent_volume_claim=V1PersistentVolumeClaimVolumeSource(claim_name=pvc_name)
        ),
    ],
    volume_mounts=[
        V1VolumeMount(name="train-scripts", mount_path=SCRIPT_MOUNT_PATH, read_only=True),
        V1VolumeMount(name="shared-data", mount_path=SHARED_DATA_MOUNT_PATH),
    ],
)

print(f"Algorithm: {algorithm}")
print(f"Cluster config: head={head_accels}, workers={n_workers}, worker_accels={worker_accels}")
print(f"Image: {ray_image}")
print(f"Script ConfigMap '{script_configmap}' mounted at {SCRIPT_MOUNT_PATH}")
print(f"Shared PVC '{pvc_name}' mounted at {SHARED_DATA_MOUNT_PATH}")
print(f"Model available at {SHARED_DATA_MOUNT_PATH}/model")

In [ ]:
entrypoint = f"sh -c 'python {SCRIPT_MOUNT_PATH}/{algorithm}_train.py'"
print(f"Entrypoint: {entrypoint}")

In [ ]:
# Newer codeflare-sdk versions auto-detect the in-cluster service
# account, which lacks RBAC to create RayJobs. Authenticate explicitly
# with the notebook user's bearer token (the test grants that user the
# admin ClusterRole in this namespace) and pass the client to RayJob.
from kubernetes import client as _k8s_client
_configuration = _k8s_client.Configuration()
_configuration.host = openshift_api_url
_configuration.api_key = {"authorization": f"Bearer {kubernetes_user_bearer_token}"}
_configuration.verify_ssl = False
_api_client = _k8s_client.ApiClient(_configuration)

job = RayJob(
    job_name=f"{algorithm}-e2e-test",
    entrypoint=entrypoint,
    cluster_config=cluster_config,
    namespace=namespace,
    ttl_seconds_after_finished=60,
    api_client=_api_client,
)

# The codeflare SDK pins a hardcoded RAY_VERSION constant and rejects any
# cluster image whose Ray version differs exactly. Since this notebook runs
# against arbitrary training hub images (which may be newer than the SDK),
# bypass that advisory client-side check; real compatibility is proven by
# the job actually running to completion below.
import codeflare_sdk.ray.rayjobs.rayjob as _rayjob_module
_rayjob_module.validate_ray_version_compatibility = (
    lambda image_name, **_: (True, False, "Ray version check bypassed")
)

job.submit()
print(f"RayJob submitted to namespace '{namespace}'")

In [ ]:
print("Waiting for job to complete...")
final_status = None
max_polls = 360
poll_interval = 10

for i in range(max_polls):
    try:
        status, ready = job.status(print_to_console=False)
        status_name = status.name if hasattr(status, 'name') else str(status)
        print(f"[{i * poll_interval}s] Status: {status_name}")
        if status_name in ("SUCCEEDED", "COMPLETE"):
            final_status = "SUCCEEDED"
            break
        if status_name == "FAILED":
            final_status = "FAILED"
            break
    except Exception as e:
        print(f"[{i * poll_interval}s] Error checking status: {e}")
    time.sleep(poll_interval)

if final_status is None:
    final_status = "TIMEOUT"

print(f"\nFinal status: {final_status}")

In [ ]:
if final_status == "SUCCEEDED":
    print(f"{algorithm.upper()} E2E test passed")
else:
    raise RuntimeError(f"{algorithm.upper()} E2E test failed with status: {final_status}")

In [ ]:
print("Notebook execution complete")